# 矩形框提示分割

## 安装依赖

In [9]:
%pip install segment-geospatial
%pip install leafmap
%pip install localtileserver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 593.1/593.1 kB 48.7 MB/s eta 0:00:00


In [16]:
import leafmap

from samgeo import SamGeo
from samgeo.common import tms_to_geotiff

## 创建交互地图



In [18]:
m = leafmap.Map(center=[-22.17615, -51.253043], zoom=18, height="800px")
m.add_basemap("SATELLITE")
m

Map(center=[-22.17615, -51.253043], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## 下载影像

In [19]:
bbox = m.user_roi_bounds()
if bbox is None:
    bbox = [-51.2565, -22.1777, -51.2512, -22.175]

In [20]:
image = "Image.tif"
tms_to_geotiff(output=image, bbox=bbox, zoom=19, source="Satellite", overwrite=True)

Downloaded image 01/45
Downloaded image 02/45
Downloaded image 03/45
Downloaded image 04/45
Downloaded image 05/45
Downloaded image 06/45
Downloaded image 07/45
Downloaded image 08/45
Downloaded image 09/45
Downloaded image 10/45
Downloaded image 11/45
Downloaded image 12/45
Downloaded image 13/45
Downloaded image 14/45
Downloaded image 15/45
Downloaded image 16/45
Downloaded image 17/45
Downloaded image 18/45
Downloaded image 19/45
Downloaded image 20/45
Downloaded image 21/45
Downloaded image 22/45
Downloaded image 23/45
Downloaded image 24/45
Downloaded image 25/45
Downloaded image 26/45
Downloaded image 27/45
Downloaded image 28/45
Downloaded image 29/45
Downloaded image 30/45
Downloaded image 31/45
Downloaded image 32/45
Downloaded image 33/45
Downloaded image 34/45
Downloaded image 35/45
Downloaded image 36/45
Downloaded image 37/45
Downloaded image 38/45
Downloaded image 39/45
Downloaded image 40/45
Downloaded image 41/45
Downloaded image 42/45
Downloaded image 43/45
Downloaded 

In [ ]:
# image = '/path/to/your/own/image.tif'

显示影像

In [21]:
m.layers[-1].visible = False
m.add_raster(image, layer_name="Image")
m

Map(bottom=37796022.0, center=[-22.17635, -51.25385], controls=(ZoomControl(options=['position', 'zoom_in_text…

## 初始化SAM类

In [22]:
sam = SamGeo(
    model_type="vit_h",
    automatic=False,
    sam_kwargs=None,
)

Model checkpoint for vit_h not found.


Downloading...
From: https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
To: /root/.cache/torch/hub/checkpoints/sam_vit_h_4b8939.pth
100%|██████████| 2.56G/2.56G [00:16<00:00, 160MB/s]


定义分割影像

In [23]:
sam.set_image(image)

显示地图。使用绘图工具，在你想要标记出来的物体周围画出矩形框，比如树木、建筑物等。

In [24]:
m

Map(bottom=9449316.0, center=[-22.17635, -51.25385], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## 创建边界框

如果没有绘制任何矩形区域，那么将使用默认的边界框，具体如下：

In [ ]:
if m.user_rois is not None:
    boxes = m.user_rois
else:
    boxes = [
        [-51.2546, -22.1771, -51.2541, -22.1767],
        [-51.2538, -22.1764, -51.2535, -22.1761],
    ]

## 分割影像

使用 predict() 方法，可以根据指定的边界框来对图像进行分割。 boxes 参数可以接受以下格式的边界框坐标列表：[[左边距, 下边距, 右边距, 上边距], [左边距, 下边距, 右边距, 上边距], …]；也可以接受 GeoJSON 格式的词典，或是指向 GeoJSON 文件的路径。

In [ ]:
sam.predict(boxes=boxes, point_crs="EPSG:4326", output="mask.tif", dtype="uint8")

## 显示结果

In [25]:
m.add_raster("mask.tif", cmap="viridis", nodata=0, layer_name="Mask")
m

OSError: Path does not exist: /content/mask.tif

## 使用现有的矢量文件作为框状提示的模板。

也可以指定矢量文件的路径

In [ ]:
url = "https://opengeos.github.io/data/sam/tree_boxes.geojson"
geojson = "tree_boxes.geojson"
leafmap.download_file(url, geojson)

展示数据

In [ ]:
m = leafmap.Map()
m.add_raster("Image.tif", layer_name="image")
style = {
    "color": "#ffff00",
    "weight": 2,
    "fillColor": "#7c4185",
    "fillOpacity": 0,
}
m.add_vector(geojson, style=style, zoom_to_layer=True, layer_name="Bounding boxes")
m

## 矩形框提示的图像分割

使用指定的矢量掩膜文件来对图像进行分割。

In [ ]:
sam.predict(boxes=geojson, point_crs="EPSG:4326", output="mask2.tif", dtype="uint8")

在地图上显示这些分割后的掩膜区域。

In [ ]:
m.add_raster("mask2.tif", cmap="Greens", nodata=0, opacity=0.5, layer_name="Tree masks")
m

![](https://i.imgur.com/SpA2NV9.gif)